In [1]:
import pandas as pd
from AgentLangGraph_clean_editor import MultiAgentAnalysisSystem
# try:
#     mp.set_start_method('spawn')
# except RuntimeError:
#     pass
results = pd.read_csv(r"C:\Users\Preet Lodaya\MLOps_Bot\Results\CyleOverCycle_forecast_select_articles.csv", parse_dates=['date'])
results['month'] = results['date'].dt.to_period('M')
results['quarter'] = results['date'].dt.to_period('Q')
results['year'] = results['date'].dt.to_period('Y')
results.drop('intersection', axis=1, inplace=True)
intersection_level = ["Store","Product"]
forecast_date = '2012-04-26'
df_dict = {
    "Forecasts_Actuals_merged": {
        "DataFrame": results,
        "Description": f"""Dataframe containing Actuals and forecasts (generated at {','.join(intersection_level)}) merged. Date of forecast generation is present in the column
                        'forecast_gen_date'. It contains forecasts generated over muliple cycles concatenated. Predictions during training period also provided."""
    }
}
obj = "To find the intersections which might have data drift issues in the test period. Use accuracy numbers to determine this. Analyze and elaborate on the nature of data drift in the top intersection you find."

In [2]:
testing = False
if testing:
    maas = MultiAgentAnalysisSystem(dataframes = df_dict, intersection_level = intersection_level, objective = obj,
                                    supervisor_llm = "gpt-5-mini", executor_llm = "gpt-5-mini", programmer_llm = "gpt-4.1-mini")
    
else:
    maas = MultiAgentAnalysisSystem(dataframes = df_dict, intersection_level = intersection_level, objective = obj)

In [3]:

from langchain_community.callbacks import get_openai_callback
with get_openai_callback() as cb:
    result = maas.analyze()
    print(f"Total Tokens: {cb.total_tokens}")
    print(f"Prompt Tokens: {cb.prompt_tokens}")
    print(f"Completion Tokens: {cb.completion_tokens}")
    print(f"Total Cost (USD): ${cb.total_cost}")


👔 SUPERVISOR NODE INVOKED
Iteration: 0/5

✅ Supervisor Decision:
   - Next Step: DataAnalyst
   - Reasoning: First, confirm the available dataframe(s) and their column names to ensure we work with the right fields and types before computing any accuracy-based drift indicators....
   - Tasks Delegated: Use list_dfs_tool to list available dataframes and show their columns and brief info for each.
📋 Tasks to Execute: 1 tasks

🔍 DATA ANALYST NODE INVOKED
Current Task: Use list_dfs_tool to list available dataframes and show their columns and brief info for each.
Iteration: 0/10
💭 Analyst Response Content:

🛠️  Tool Calls Detected: 1
   - Tool: list_dfs_tool
Available dataframes: dict_keys(['Forecasts_Actuals_merged'])

🔍 DATA ANALYST NODE INVOKED
Iteration: 1/10
💭 Analyst Response Content:
Plan:
- Query the DataFrame store to list available DataFrames and capture their shapes, column names and descriptions.
- Report the results (columns and brief info) so they can be used for downstream an

In [7]:
print(result['messages'][0].content)

REASONING: Goal: Frame the problem and confirm basic structure so we can isolate the true test-period records (date > forecast_gen_date) and know how many cycles/intersections we have. This ensures our drift detection will be done correctly at the Store-Product level using all cycles and test-only records.
Next Step: DataAnalyst
TASKS DELEGATED: From Forecasts_Actuals_merged, report: 1) number of unique Stores, Products, and Store-Product intersections; 2) number of unique forecast cycles (unique forecast_gen_date) and their min/max; 3) min/max of target date; 4) counts of records where date > forecast_gen_date (test) vs date <= forecast_gen_date (train-ish). Return only these summary numbers.


In [4]:
result['dataframe_info']['Forecasts_Actuals_merged']

{'Summary': 'DATASET: Results (35050 rows × 14 columns) Columns Names: forecast_gen_date, date, actual_sales, Country, Store, Division, Category, Product, forecast_es, forecast_sarima, forecast_gen_date\n, month, quarter, year\n\n',
 'Description': "Dataframe containing Actuals and forecasts (generated at Store,Product) merged. Date of forecast generation is present in the column\n                        'forecast_gen_date'. It contains forecasts generated over muliple cycles concatenated. Predictions during training period also provided."}

In [8]:
results.loc[results['date'] > results['forecast_gen_date']]['forecast_sarima'].describe()

count      9950.000000
mean      51080.489282
std       29935.695498
min        6485.615369
25%       31162.479452
50%       44342.838240
75%       63812.293625
max      602646.463700
Name: forecast_sarima, dtype: float64

In [10]:
import json

def custom_serializer(obj):
    """Custom serializer for objects not serializable by default."""
    if hasattr(obj, "__dict__"):
        return obj.__dict__  # Convert objects with __dict__ to a dictionary
    return str(obj)  # Fallback: convert to string

# Save the result to a file
with open("Results/final_state.txt", "w") as f:
    json.dump(result, f, indent=4, default=custom_serializer)

In [4]:
print(result['query_response'])

Requirement: Identify Store–Product intersections that likely experienced data drift in the test period using accuracy numbers (absolute error at the intersection level).

Executive summary
- Scope: 50 Store–Product intersections evaluated over a train period vs test period (test start: 2012-04-26).
- Drift signal used: drift_ratio = MAE_test_selected / MAE_train_selected (using the model that had lower training MAE per intersection). A ratio > 1 indicates deterioration; we also assess scale-normalized impact via test_mae_pct_of_mean = MAE_test_selected / mean_actual_test.
- Overall distribution (50 intersections):
  - Median drift_ratio: 1.785; 75th percentile: 2.506; 90th: 3.677; 95th: 3.975.
  - 21 intersections (42%) show severe deterioration (drift_ratio ≥ 2.0).
  - 38 intersections (76%) have notable deterioration (drift_ratio ≥ 1.5).
  - 2 intersections improved (drift_ratio < 1).

Drift candidates to investigate (prioritized)
- Triage logic:
  - Tier 1 (Severe and high-impact):

In [2]:
print(result['messages'][-1].content)

REASONING: We now know the evaluation window, intersection counts, and per-(Store, Product) error metrics across the test period. We can prioritize “easy” accuracy gains by: (a) switching to the better of ES vs SARIMA where their SAE gap is largest (immediate, low-effort wins), and (b) focusing deeper modeling on the intersections that contribute the most to total residual error (best-of-two SAE), especially those with high normalized MAE (greater headroom). The results below provide a concise, data-driven shortlist for action.


In [3]:
print(result['query_response'])

Requirement: Identify the store–product intersections to focus on to improve overall forecasting accuracy easily.

Scope and baseline
- Test window: 2012-04-26 to 2013-04-14; 50 intersections; 52 observations per intersection (weekly).
- Global test SAE totals:
  - ES total SAE: 6,338,640.02
  - SARIMA total SAE: 6,469,354.01
  - Best-of-two total SAE (pick the better model per intersection): 5,339,270.93
  - Win counts: ES better on 20 intersections; SARIMA better on 30 intersections.
- Immediate aggregate improvement from model selection:
  - vs ES: −999,369.09 SAE (−15.8%)
  - vs SARIMA: −1,130,083.09 SAE (−17.5%)

A) Quick wins: switch to the better model where the ES–SARIMA gap (delta_SAE) is largest
- These deliver the largest immediate error reductions by simply adopting the better model at each intersection.
Top 10 by delta_SAE (descending); recommended model in parentheses:
1) Store 10, Product 5 — delta_SAE 324,725.09 (use SARIMA)
2) Store 20, Product 7 — delta_SAE 261,460.63